# 00 · End-to-end: game video → 2D court tracking and touch-by-touch data

Runs the CourtVision pipeline on one video in Colab:

1. detect + track players, referees, ball and hoop (every video frame)
2. court keypoints → homography, carried through camera pans by optical flow
3. clean the ball track (select, de-outlier, interpolate)
4. assign teams by jersey colour
5. ball possession with hysteresis
6. passes, interceptions, turnovers, shots (made/missed, 2PT/3PT), rebounds
7. court positions in metres for every player, speed and distance
8. the ball's court position: the holder's spot, the straight ground line of a pass or shot, or the floor
9. **`tracking_2d`**: x, y in metres for every on-court player and the ball, resampled to 10 Hz
10. **touch-by-touch table**: one row per touch, with every other player's position and speed

Weights come from notebooks 01 and 02, saved in `MyDrive/courtvision/models/`. The earlier 3D
reconstruction is still in the package: set `cfg.output.mode = "3d"` to get `scene.json` and the
3D viewer as well.

In [ ]:
# Runtime > Change runtime type > GPU before running.
!nvidia-smi -L
from google.colab import drive
drive.mount("/content/drive")

# Everything that must survive a Colab disconnect (runs, weights) lives in Drive.
DRIVE_ROOT = "/content/drive/MyDrive/courtvision"
import os
os.makedirs(f"{DRIVE_ROOT}/models", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)

In [ ]:
REPO_URL = "https://github.com/moorexavierb/CourtVision.git"
BRANCH = "main"   # set to a PR branch to test it, then Runtime > Restart session and run all
!rm -rf /content/courtvision && git clone -q -b {BRANCH} {REPO_URL} /content/courtvision
%cd /content/courtvision
!pip -q install -e ".[models]"

## Configure

In [ ]:
from courtvision import Config, Pipeline

cfg = Config.load("configs/default.yaml")
cfg.models.detector = f"{DRIVE_ROOT}/models/player_detector.pt"
cfg.models.court_keypoints = f"{DRIVE_ROOT}/models/court_keypoints.pt"
if os.path.exists(f"{DRIVE_ROOT}/models/ball_detector.pt"):
    cfg.models.ball_detector = f"{DRIVE_ROOT}/models/ball_detector.pt"
cfg.teams.team_names = ["Lakers", "Spurs"]   # display names; team 0/1 is decided by jersey colour
cfg.output.dir = f"{DRIVE_ROOT}/output"
cfg.output.cache_dir = f"{DRIVE_ROOT}/stubs"  # detections are cached; re-runs skip the GPU pass

cfg.output.mode = "2d"            # "3d" adds the camera, ball height, scene.json and the 3D viewer
cfg.output.tracking_fps = 10      # samples per second in tracking_2d (None = every video frame)

## Pick a video (upload, or put it in `MyDrive/courtvision/input_videos/`)

In [ ]:
import glob
videos = sorted(glob.glob(f"{DRIVE_ROOT}/input_videos/*.mp4"))
if not videos:
    from google.colab import files
    up = files.upload()
    videos = list(up)
VIDEO = videos[0]
VIDEO

## Run

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
pipe = Pipeline(cfg)
result = pipe.run(VIDEO)
OUT = os.path.join(cfg.output.dir, os.path.splitext(os.path.basename(VIDEO))[0])
print(f"{len(result.players)} frames · {len(result.team_of)} player tracks · "
      f"{len(result.possessions)} touches · {len(result.events)} events")

## Diagnostics: which stage is weak?

If the positions look wrong, start here. The report checks each stage on this clip: how often the court is mapped (from keypoints, or carried by camera motion between them) and how far off the keypoint fits are, how many player tracks break up or land off the court, how cleanly the two jersey colours separate, and how often the ball is seen and placed on the court. Problems are listed weakest stage first, each with what would fix it.

In [ ]:
from courtvision.diagnostics import format_report
from IPython.display import Image
print(format_report(result.diagnostics))
Image(f"{OUT}/diagnostics.png")

## 2D tracking: x, y for every player and the ball, 10 times a second

`tracking_2d.csv` / `.parquet`, one row per sample and object. Court axes: `x_m` along the
sideline (0 to 28.65 m, baseline to baseline), `y_m` across (0 to 15.24 m).

| Column | |
|---|---|
| `sample`, `time_s`, `frame` | sample number, time on the 10 Hz clock, nearest video frame |
| `object`, `id`, `team` | `player` (track id, team) or `ball` (id -1) |
| `x_m`, `y_m`, `speed_kmh`, `has_ball` | position, speed, whether this player holds the ball |
| `source` | players: `tracked`. Ball: `held` (at the holder), `flight` (straight ground line of a pass or shot), `floor` (loose ball, projected directly) |

In [ ]:
import pandas as pd
from courtvision.tracking2d import coverage
t2d = pd.read_parquet(f"{OUT}/tracking_2d.parquet")
print(coverage(t2d))
display(t2d.head(12))
# Players per sample: 10 means everyone on the court is placed.
t2d[t2d.object == "player"].groupby("sample").size().value_counts().sort_index()

In [ ]:
# Top-down replay of the first 20 seconds (one frame per sample).
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from courtvision.court.geometry import get_court
court = get_court(cfg.court.standard)
colors = ["#%02x%02x%02x" % tuple(reversed(c)) for c in pipe.team_colors_bgr]
fig, ax = plt.subplots(figsize=(9, 5))
def draw(s):
    ax.clear(); ax.set_xlim(-1, court.length + 1); ax.set_ylim(-1, court.width + 1); ax.set_aspect("equal")
    ax.add_patch(plt.Rectangle((0, 0), court.length, court.width, fill=False))
    ax.axvline(court.length / 2, color="k", lw=0.8)
    for hx, hy, _ in court.hoops:
        ax.add_patch(plt.Circle((hx, hy), court.three_radius, fill=False, lw=0.8))
    d = t2d[t2d["sample"] == s]
    for _, r in d[d.object == "player"].iterrows():
        team = cfg.teams.team_names.index(r.team) if r.team in cfg.teams.team_names else -1
        ax.scatter(r.x_m, r.y_m, s=160, c=colors[team] if team >= 0 else "grey", edgecolors="k")
        ax.text(r.x_m, r.y_m, str(r.id), ha="center", va="center", fontsize=7)
    b = d[d.object == "ball"]
    if len(b):
        ax.scatter(b.x_m, b.y_m, s=50, c="orange", edgecolors="k", zorder=5)
    ax.set_title(f"t = {s / cfg.output.tracking_fps:.1f} s")
samples = sorted(t2d["sample"].unique())[: int(20 * cfg.output.tracking_fps)]
anim = animation.FuncAnimation(fig, draw, frames=samples, interval=1000 / cfg.output.tracking_fps)
plt.close(fig)
HTML(anim.to_jshtml())

## Touch-by-touch data

One row per touch: a player's continuous control of the ball, from catch to release.

| Columns | What they hold |
|---|---|
| `touch_id`, `possession_id`, `start_frame`/`end_frame`, `start_s`/`end_s`, `duration_s` | when; `possession_id` goes up each time the other team gets the ball |
| `player_id`, `team`, `jersey`, `name` | who (jersey and name stay empty until a roster mapping is set, see below) |
| `start_action` | how the touch began: `catch`, `steal`, `offensive_rebound`, `defensive_rebound`, `recovery`, `start_of_clip` |
| `action`, `outcome`, `target_id`, `target_name` | what they did: `pass` (completed / intercepted, with the receiver), `shot` (made / missed), `turnover`, `loose_ball`, `fumble`, `end_of_clip` |
| `dribbles`, `shot_points`, `shot_distance_m`, `pass_air_s` | detail on the action |
| `start_x_m`, `start_y_m`, `x_m`, `y_m`, `speed_kmh`, `max_speed_kmh`, `distance_covered_m`, `dist_to_basket_m` | the ball handler, in court metres and km/h; `x_m`/`y_m`/`speed_kmh` are at the action frame |
| `nearest_defender_id`, `nearest_defender_m`, `defenders_within_2m`, `players_tracked` | pressure on the ball |
| `tm1_*` … `tm4_*`, `op1_*` … `op5_*` | every other player at the action frame (teammates, then opponents, nearest first): `id`, `x_m`, `y_m`, `speed_kmh`, `dist_m` to the handler |
| `confidence` | 0 to 1: detector confidence, court calibration coverage, touch length, and whether the start and end matched a detected event |

Broadcast cameras rarely show all ten players, so the slots for players off screen are empty.

**Names and jerseys:** set `pipe.player_info = {track_id: {"jersey": "23", "name": "LeBron James"}}` before `pipe.run`, or rebuild the table afterwards with the cell below.

In [ ]:
import pandas as pd
touches = pd.read_parquet(f"{OUT}/touches.parquet")
core = ["touch_id", "possession_id", "start_s", "player_id", "team", "start_action", "action", "outcome",
        "target_id", "dribbles", "x_m", "y_m", "speed_kmh", "nearest_defender_m", "confidence"]
print(f"{len(touches)} touches, {touches.shape[1]} columns")
display(touches[core].head(30))

In [ ]:
# Everyone on the floor at one touch: the ball handler plus each teammate and opponent slot.
t = touches.iloc[0]
slots = [("handler", t.player_id, t.x_m, t.y_m, t.speed_kmh, 0.0)]
for prefix, n in (("tm", 4), ("op", 5)):
    for k in range(1, n + 1):
        if pd.notna(t[f"{prefix}{k}_id"]):
            slots.append((f"{prefix}{k}", *(t[f"{prefix}{k}_{c}"] for c in ("id", "x_m", "y_m", "speed_kmh", "dist_m"))))
print(f"touch {t.touch_id}: player {t.player_id} {t.start_action} -> {t.action} ({t.outcome})")
display(pd.DataFrame(slots, columns=["slot", "player_id", "x_m", "y_m", "speed_kmh", "dist_to_handler_m"]))

In [ ]:
# Optional: add names once you know which track is which player, then rewrite the touches files.
from courtvision.events.touches import touches_table
pipe.player_info = {}   # e.g. {7: {"jersey": "23", "name": "LeBron James"}}
if pipe.player_info:
    touches = touches_table(result.possessions, result, cfg.teams.team_names, pipe.court, pipe.player_info)
    touches.to_parquet(f"{OUT}/touches.parquet", index=False)
    touches.to_csv(f"{OUT}/touches.csv", index=False)

In [ ]:
# Per-player summary of the touches, then the event log and box score.
display(pd.crosstab([touches.player_id, touches.team], touches.action, margins=True))
events = pd.read_csv(f"{OUT}/events.csv")
display(events.head(30))
display(pd.read_csv(f"{OUT}/box_score.csv"))

## Annotated video

In [ ]:
# Re-encode to H.264 so the browser can play it inline.
src = f"{OUT}/{os.path.splitext(os.path.basename(VIDEO))[0]}_annotated.mp4"
!ffmpeg -y -loglevel error -i "{src}" -vcodec libx264 -crf 26 /content/preview.mp4
from IPython.display import Video
Video("/content/preview.mp4", embed=True, width=960)

## Downloads

In [ ]:
display(touches.head(10))
from google.colab import files
files.download(f"{OUT}/tracking_2d.csv")
files.download(f"{OUT}/touches.csv")